# Project 9: Calgary Business Survival Analyzer -- Exploratory Data Analysis

This notebook explores the Calgary Business Licences dataset (22,000+ records) to understand:

1. Business type distributions
2. Survival time analysis
3. Community-level patterns
4. Feature engineering for modelling

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Add project root to path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_and_preprocess, get_community_summary

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)

print("Setup complete.")

## 1. Data Loading

Fetch business-licence and civic-census data from Calgary Open Data (or load from cache).

In [ ]:
licences, census = load_and_preprocess()

print(f"Business Licences: {licences.shape[0]:,} rows x {licences.shape[1]} columns")
print(f"Civic Census:      {census.shape[0]:,} rows x {census.shape[1]} columns")
licences.head()

In [ ]:
# Column overview
licences.info()

In [ ]:
# Basic statistics
licences.describe(include="all").T

In [ ]:
# Missing values
missing = licences.isnull().sum()
missing_pct = (missing / len(licences) * 100).round(2)
pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct}).query("missing_count > 0").sort_values("missing_pct", ascending=False)

## 2. Business Type Distributions

Which types of businesses are most common in Calgary?

In [ ]:
type_counts = licences["business_category"].value_counts().head(20).reset_index()
type_counts.columns = ["Business Type", "Count"]

fig = px.bar(
    type_counts,
    x="Count",
    y="Business Type",
    orientation="h",
    title="Top 20 Business Types by Number of Licences",
    color="Count",
    color_continuous_scale="Viridis",
)
fig.update_layout(yaxis=dict(autorange="reversed"), height=600, showlegend=False)
fig.show()

In [ ]:
# Survival rate by business type
type_survival = (
    licences.groupby("business_category")
    .agg(count=("getbusid", "count"), survival_rate=("survived", "mean"))
    .reset_index()
    .query("count >= 20")
    .sort_values("survival_rate", ascending=True)
)

fig = px.bar(
    type_survival.tail(20),
    x="survival_rate",
    y="business_category",
    orientation="h",
    title="Survival Rate by Business Type (min 20 businesses)",
    color="survival_rate",
    color_continuous_scale="RdYlGn",
    labels={"survival_rate": "Survival Rate", "business_category": "Business Type"},
)
fig.update_layout(yaxis=dict(autorange="reversed"), height=600, showlegend=False)
fig.show()

In [ ]:
# Status distribution
status_counts = licences["jobstatusdesc"].str.title().value_counts().reset_index()
status_counts.columns = ["Status", "Count"]

fig = px.pie(
    status_counts,
    names="Status",
    values="Count",
    title="Business Licence Status Distribution",
    hole=0.4,
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.show()

In [ ]:
# Home-occupation vs. commercial
home_occ = licences.groupby("is_home_occupation").agg(
    count=("getbusid", "count"),
    survival_rate=("survived", "mean"),
    avg_age_days=("business_age_days", "mean"),
).reset_index()
home_occ["is_home_occupation"] = home_occ["is_home_occupation"].map({0: "Commercial", 1: "Home Occupation"})
home_occ

## 3. Survival Time Analysis

How long do businesses survive? What does the distribution of business age look like?

In [ ]:
# Distribution of business age (in years)
licences["business_age_years"] = licences["business_age_days"] / 365.0

fig = px.histogram(
    licences,
    x="business_age_years",
    nbins=50,
    title="Distribution of Business Age (Years)",
    labels={"business_age_years": "Business Age (Years)"},
    color_discrete_sequence=["steelblue"],
)
fig.update_layout(height=400)
fig.show()

In [ ]:
# Business age by status
fig = px.box(
    licences,
    x="jobstatusdesc",
    y="business_age_years",
    title="Business Age Distribution by Status",
    labels={"jobstatusdesc": "Status", "business_age_years": "Age (Years)"},
    color="jobstatusdesc",
)
fig.update_layout(height=450, showlegend=False)
fig.show()

In [ ]:
# Kaplan-Meier survival curves by business type
from src.model import SurvivalAnalyzer

analyzer = SurvivalAnalyzer()

# Create event_observed: 1 if closed, 0 if still active (censored)
licences["event_observed"] = 1 - licences["survived"]

km_curves = analyzer.kaplan_meier_by_group(
    licences,
    duration_col="business_age_days",
    event_col="event_observed",
    group_col="business_category",
    top_n=8,
)

km_curves["timeline_years"] = km_curves["timeline"] / 365.0

fig = px.line(
    km_curves,
    x="timeline_years",
    y="survival_probability",
    color="label",
    title="Kaplan-Meier Survival Curves by Business Type",
    labels={
        "timeline_years": "Years Since Licence Issued",
        "survival_probability": "Survival Probability",
        "label": "Business Type",
    },
)
fig.update_layout(height=500)
fig.show()

In [ ]:
# Survival rates at 1, 3, and 5 years
milestone_df = analyzer.survival_rates_at_milestones(
    licences,
    duration_col="business_age_days",
    event_col="event_observed",
    group_col="business_category",
    top_n=15,
)
milestone_df

In [ ]:
# New businesses per year
yearly = licences.groupby("issue_year").size().reset_index(name="new_licences")

fig = px.bar(
    yearly,
    x="issue_year",
    y="new_licences",
    title="New Business Licences Issued per Year",
    labels={"issue_year": "Year", "new_licences": "New Licences"},
    color_discrete_sequence=["darkorange"],
)
fig.update_layout(height=400)
fig.show()

## 4. Community Patterns

Which communities are business-dense? Which have the highest (and lowest) survival rates?

In [ ]:
community = get_community_summary(licences)
print(f"Communities analysed: {len(community)}")
community.head(20)

In [ ]:
# Top 20 communities by number of businesses
fig = px.bar(
    community.head(20),
    x="comdistnm",
    y="total_businesses",
    color="survival_rate",
    color_continuous_scale="RdYlGn",
    title="Top 20 Communities by Total Businesses (coloured by survival rate)",
    labels={"comdistnm": "Community", "total_businesses": "Total Businesses", "survival_rate": "Survival Rate"},
)
fig.update_layout(xaxis_tickangle=-45, height=500)
fig.show()

In [ ]:
# Survival rate vs. business diversity
fig = px.scatter(
    community,
    x="business_diversity",
    y="survival_rate",
    size="total_businesses",
    hover_name="comdistnm",
    title="Community Survival Rate vs. Business Diversity",
    labels={
        "business_diversity": "Number of Distinct Business Categories",
        "survival_rate": "Survival Rate",
        "total_businesses": "Total Businesses",
    },
    color="survival_rate",
    color_continuous_scale="RdYlGn",
)
fig.update_layout(height=500)
fig.show()

In [ ]:
# Home occupation percentage across communities
fig = px.bar(
    community.sort_values("home_occupation_pct", ascending=False).head(20),
    x="comdistnm",
    y="home_occupation_pct",
    title="Top 20 Communities by Home-Occupation Business Percentage",
    labels={"comdistnm": "Community", "home_occupation_pct": "Home Occupation %"},
    color_discrete_sequence=["mediumseagreen"],
)
fig.update_layout(xaxis_tickangle=-45, height=450)
fig.show()

In [ ]:
# Correlation heatmap of numeric features
numeric_cols = [
    "business_age_days", "survived", "is_home_occupation",
    "business_count", "business_diversity", "avg_business_age",
    "issue_year", "issue_month",
]
available = [c for c in numeric_cols if c in licences.columns]
corr = licences[available].corr()

fig = px.imshow(
    corr,
    text_auto=".2f",
    title="Feature Correlation Heatmap",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
)
fig.update_layout(height=550, width=650)
fig.show()

## Summary

Key observations:

- The dataset contains 22,000+ business-licence records spanning multiple years.
- Business survival rates vary significantly by licence type and community.
- Communities with greater business diversity tend to have higher survival rates.
- Home-occupation businesses exhibit distinct survival patterns.
- Temporal trends (year of issue) may capture economic cycles affecting survival.

These insights inform the survival-analysis and classification models built in `src/model.py` and visualised in `app.py`.